In [ ]:
"""Figure 4B: group-mean theta and gamma scalp power topographies.

Data source: TFR_CONCATENATED_*.npy arrays from the original Figure 4A,
shape (participants, channels, frequencies, time). These arrays should
ALREADY contain baseline-normalised power (z-scores).

Individual *-power-tfr.h5 files are read ONLY for channel positions and the
condition-specific frequency and time axes. No re-normalisation is performed.

Run in a Python environment with numpy, matplotlib and MNE-Python installed.
"""

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mne

# ---------------------------------------------------------------------
# 1. EDIT PATHS / SETTINGS
# ---------------------------------------------------------------------
DATA_DIR = Path(r"F:\research project\# study 1\eeg_data\set\Y")

# IMPORTANT: point this at the directory containing TFR_CONCATENATED_Y_*.npy.
# If your notebook already defines tfr_group_dir, you can instead set:
# GROUP_TFR_DIR = Path(tfr_group_dir)
GROUP_TFR_DIR = DATA_DIR / "group_results"  # CHANGE if needed

OUTPUT_DIR = GROUP_TFR_DIR / "Figure4B_topoplots"
REFERENCE_SUBJECT = "s1_pac_sub01"

BANDS = {
    "Theta (4–8 Hz)": (4.0, 8.0),
    "Gamma (30–80 Hz)": (30.0, 80.0),
}

WINDOWS = {
    "0–200 ms": (0.0, 0.2),
    "200–450 ms": (0.2, 0.45),
}

# Rows: FTT, De-CRAT baseline, De-CRAT adaptation.
# Each row has planning early/late and execution early/late.
CONDITIONS = [
    ("FTT", "BL", None),
    ("De-CRAT baseline", "MAIN", "baseline"),
    ("De-CRAT adaptation", "MAIN", "adaptation"),
]
STAGES = [("Planning", "plan"), ("Execution", "go")]

CMAP = "plasma"   # Same as original Figure 4A
VLIM = (-0.3, 1.0)
DPI = 300


# ---------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------------------
def naming(task, stage_code, block):
    """Return the original group-array name and individual H5 suffix."""
    stem = f"{task}_{stage_code}"
    if block is not None:
        stem += f"_{block}"
    return f"Y_{stem}", f"_{stem}_power-tfr.h5"


def read_reference_tfr(task, stage_code, block):
    """Read an individual TFR for channel metadata and exact time axis."""
    _, suffix = naming(task, stage_code, block)
    path = (
        DATA_DIR / REFERENCE_SUBJECT / "preproc" / "analysis"
        / f"{REFERENCE_SUBJECT}{suffix}"
    )
    if not path.exists():
        raise FileNotFoundError(f"Reference TFR not found: {path}")

    tfr = mne.time_frequency.read_tfrs(str(path), verbose=False)
    if isinstance(tfr, list):
        if len(tfr) != 1:
            raise ValueError(f"Expected exactly one TFR in {path}")
        tfr = tfr[0]

    # Important: do not call apply_baseline here.
    tfr = tfr.copy().pick("eeg")
    return tfr


def load_group_topographies():
    """Average pre-normalised data over participants, frequency and time."""
    topographies = {}
    reference_info = None
    reference_channels = None

    for row_label, task, block in CONDITIONS:
        for stage_label, stage_code in STAGES:
            name, _ = naming(task, stage_code, block)
            npy_path = GROUP_TFR_DIR / f"TFR_CONCATENATED_{name}.npy"
            if not npy_path.exists():
                raise FileNotFoundError(
                    f"Group array not found: {npy_path}\n"
                    "Set GROUP_TFR_DIR to your existing tfr_group_dir."
                )

            tfr = read_reference_tfr(task, stage_code, block)
            freqs, times = tfr.freqs, tfr.times
            data = np.load(npy_path, mmap_mode="r")

            if data.ndim != 4:
                raise ValueError(
                    f"{npy_path.name}: expected [subjects, channels, freqs, time], "
                    f"got {data.shape}"
                )
            if data.shape[1:] != (len(tfr.ch_names), len(freqs), len(times)):
                raise ValueError(
                    f"Dimension mismatch for {name}: group {data.shape}, "
                    f"reference {(len(tfr.ch_names), len(freqs), len(times))}"
                )

            if reference_info is None:
                reference_info = tfr.info.copy()
                reference_channels = list(tfr.ch_names)
            elif tfr.ch_names != reference_channels:
                raise ValueError(
                    f"Channel order differs for {name}. Check creation of the .npy arrays."
                )

            print(f"{name:27s}  group shape={data.shape}")

            for band_label, (fmin, fmax) in BANDS.items():
                freq_mask = (freqs >= fmin - 1e-8) & (freqs <= fmax + 1e-8)
                if not freq_mask.any():
                    raise ValueError(f"No frequency samples for {band_label} in {name}")

                for window_label, (tmin, tmax) in WINDOWS.items():
                    # Non-overlapping, half-open windows [start, end).
                    time_mask = (times >= tmin) & (times < tmax)
                    if not time_mask.any():
                        raise ValueError(f"No samples for {window_label} in {name}")

                    subset = data[:, :, freq_mask, :][:, :, :, time_mask]
                    if not np.isfinite(subset).all():
                        raise ValueError(f"NaN or infinity detected: {name}, {band_label}, {window_label}")

                    # Mean across participants, selected frequencies, and time.
                    topo = np.asarray(subset.mean(axis=(0, 2, 3)))
                    topographies[(row_label, stage_label, band_label, window_label)] = topo

                    outside = np.mean((topo < VLIM[0]) | (topo > VLIM[1])) * 100
                    print(
                        f"  {band_label:17s} {stage_label:9s} {window_label:11s}: "
                        f"min={topo.min():+6.3f}, max={topo.max():+6.3f}, "
                        f"outside colour limits={outside:4.1f}% channels"
                    )

    return topographies, reference_info


def plot_band(topographies, info, band_label, output_stem):
    fig, axes = plt.subplots(3, 4, figsize=(12, 8.2), layout="constrained")
    windows = list(WINDOWS)
    last_image = None

FTT          | Planning  | 23 files
FTT          | Execution | 23 files
baseline     | Planning  | 23 files
baseline     | Execution | 23 files
adaptation   | Planning  | 23 files
adaptation   | Execution | 23 files


TypeError: compute_group_maps_from_npy() missing 1 required positional argument: 'info'

In [22]:
# Use the channel information from a sample file
sample_path = (
    DATA_DIR / "s1_pac_sub01" / "preproc" / "analysis"
    / "s1_pac_sub01_BL_plan_power-tfr.h5"
)
tfr_group_dir = Path(r"F:\research project\# study 1\eeg_data\set\Y group\sensors\tfr")

sample_tfr = load_tfr(sample_path)
info = sample_tfr.info

group_maps = compute_group_maps_from_npy(
    tfr_group_dir,
    info
)

fig_theta = plot_band(
    group_maps,
    info,
    "Theta (4–8 Hz)",
    "Figure4B_theta_topoplots"
)

fig_gamma = plot_band(
    group_maps,
    info,
    "Gamma (30–80 Hz)",
    "Figure4B_gamma_topoplots"
)

plt.show()

Applying baseline correction (mode: zscore)
Applying baseline correction (mode: zscore)
Applying baseline correction (mode: zscore)
Saved: F:\research project\# study 1\eeg_data\set\Y\group_results\Figure4B_topoplots\Figure4B_theta_topoplots.png
Saved: F:\research project\# study 1\eeg_data\set\Y\group_results\Figure4B_topoplots\Figure4B_theta_topoplots.svg
Theta (4–8 Hz) | FTT scale: [-1.654, 9.582]
Theta (4–8 Hz) | De-CRAT scale: [0.040, 6.026]
Saved: F:\research project\# study 1\eeg_data\set\Y\group_results\Figure4B_topoplots\Figure4B_gamma_topoplots.png
Saved: F:\research project\# study 1\eeg_data\set\Y\group_results\Figure4B_topoplots\Figure4B_gamma_topoplots.svg
Gamma (30–80 Hz) | FTT scale: [-0.289, 1.315]
Gamma (30–80 Hz) | De-CRAT scale: [0.027, 1.772]
